# Calabi–Yau Threefold Landscape — Graph Knowledge Network

Conversion of the `assets/20260806 CalabiYau3fold/` research folder into a
first-class **knowledge graph** usable by the project tooling: global graph →
depth-3 local graphs → encoder RAG → DeepSeek agents → recursive growth.

**Source:** `assets/20260806 CalabiYau3fold/ResearchReferences/`

| piece | content |
|---|---|
| `20260805 Calabi Yau Threefold.ipynb` / `.html` | the research notes: **seven target totals** $h^{\mathrm{tot}}=h^{1,1}+h^{2,1}$ (17, 28, 29, 66, 80, 81, 92) with **data-verified verdicts** (corrected 2026-08-06) |
| `papers/` + `papers/_extracted/` | 25 PDFs (classifications, constructions, 2024–2026 ML studies) + pypdf text of 20 papers (the encoder corpus) |
| `datasets/` | Kreuzer–Skarke (`alltoric` 30,108 · `wp4` 2,780 · `toric` 10,237), Davies zoo (30,389), TCI Hodge page (210) |
| moment-problem / SDP PDFs | Laurent, Schmüdgen, Josz–Henrion, de Klerk–Laurent, moment-problem book |

**What this notebook does**

1. loads the datasets with **fixed paths** (the original research notebook
   pointed at a non-existent `Bootstrap Calabi Yau 3 fold\ResearchReferences\datasets`);
2. reproduces the headline statistics (KS gap, self-mirror diagonal, per-total
   tables, 100%-toric sets, non-toric ladders);
3. builds a typed knowledge graph via **`tools/build_cy3.py`** → `graph_data/cy3/`
   (92 nodes, 133 edges, ~2,300 encoded chunks — the default GraphRAG graph is
   **untouched**, the two graphs coexist);
4. demonstrates the full stack on the CY3 domain: local graphs, vector RAG,
   `NodeAgent`, `GrowthAgent`;
5. serves the graph in the web control center:
   `python ui/server.py --graph graph_data/cy3`.

**The seven totals, decoded (verified 2026-08-06):**

| total | # known | toric | verdict |
|---|---|---|---|
| **17** | 3 | 0 | **UNIQUE** — below the KS gap (22); all 3 pairs are free quotients (π₁ ≠ 1) |
| **28** | 17 | 7 | **UNIQUE** — first self-mirror total (14,14); rigid (28,0); zero K3-fibered models |
| **29** | 8 | 6 | **MILD** — neighbour of 28; extreme (1,28), χ = −54 |
| **66** | 63 | 61 | **SPECIAL** — contains the four-quadrics CICY (1,65), root of Hua's tower |
| **80** | 77 | 75 | **GENERIC** — self-mirror generic; only the Siegel χ=80 echo |
| **81** | 76 | **76** | **NOTABLE** — 100% toric (largest below 85); double gap in the non-toric ladders |
| **92** | 89 | **89** | **NOTABLE-BUT-GENERAL** — 100% toric is the norm by r=92; Siegel χ=92 echo |


## 1. Setup & Imports

Kernel: the project's `agenti_ai` conda env (Python 3.13). The cell below
bootstraps `sys.path` to the workspace root (works from any kernel cwd), then
loads the shared tooling: `Config`, the tool registry, the two agents, and the
new `tools.build_cy3` seed module (this notebook's graph builder).

In [ ]:
from tools.config import Config
from tools.IPP import ToolRegistry
from tools.graph_tools import ensure_tools
from tools.agents import NodeAgent, GrowthAgent
from tools.build import export_backward_compatible
from tools.build_cy3 import build_cy3_graph, CY3_ROOT, CY3_OUT
from LLMs.deepseek import DeepSeekProvider, MockProvider

workspace : D:\Deepin\Programming\20260720 GraphRAG
cy3 source: D:\Deepin\Programming\20260720 GraphRAG\assets\20260806 CalabiYau3fold\ResearchReferences
cy3 output: D:\Deepin\Programming\20260720 GraphRAG\graph_data\cy3
tools     : 23 registered
llm       : deepseek-v4-flash


## 2. The Calabi–Yau Threefold Datasets

All files live in `ResearchReferences\datasets\` (downloaded 2026-08-05):

| file | content | size |
|---|---|---|
| `alltoric.spec` | **Kreuzer–Skarke:** all 473,800,776 reflexive 4-polytopes → 30,108 distinct Hodge pairs | 30,108 |
| `wp4.spec` | 7,555 hypersurfaces in weighted ℙ⁴ | 2,780 pairs |
| `toric.spec` | 184,026 IP weight systems (K3-fibered toric CY3s) | 10,237 pairs |
| `Kreuzer_TCI_hodge_data_0103214.html` | toric complete intersections (unpublished Kreuzer data) | 210 pairs parsed |
| `hodge_list_davies_*.txt` | Rhys Davies' "zoo": all pairs known in 2011 + per-pair references | 30,389 pairs |

The cell below loads them **from this workspace** (the original notebook's
`ROOT` was broken).

In [2]:
# §2 — load all datasets (paths relative to the CalabiYau3fold folder)
DATASETS = CY3_ROOT / "datasets"

def load_spec(name):
    """KS-style file: 'h11 h21 chi' per line -> {(h11,h21): chi}"""
    out = {}
    with open(DATASETS / name, encoding="utf-8") as f:
        for line in f:
            m = re.match(r"\s*(\d+)\s+(\d+)\s+(-?\d+)", line)
            if m:
                out[(int(m.group(1)), int(m.group(2)))] = int(m.group(3))
    return out

def load_davies(name):
    """Davies list: '(h11,h21)  #refs' -> {(h11,h21): refs}"""
    out = {}
    with open(DATASETS / name, encoding="utf-8", errors="replace") as f:
        for line in f:
            m = re.match(r"\((\d+),(\d+)\)\s+#?(.*)", line.strip())
            if m:
                out[(int(m.group(1)), int(m.group(2)))] = m.group(3).strip()
    return out

def load_tci():
    """Kreuzer TCI Hodge data page -> set of pairs"""
    with open(DATASETS / "Kreuzer_TCI_hodge_data_0103214.html",
              encoding="utf-8", errors="replace") as f:
        text = f.read()
    return {(int(a), int(b)) for a, b, c in
            re.findall(r"H:\s*(\d+)\s+(\d+)\s+\[\s*(-?\d+)\]", text)}

ks     = load_spec("alltoric.spec")                    # 30,108 toric pairs
wp     = load_spec("wp4.spec")                         # 2,780 weighted-P4 pairs
toric  = load_spec("toric.spec")                       # 10,237 K3-fibered pairs
davies = load_davies("hodge_list_davies_sorted_h.txt") # 30,389 'all known' pairs
tci    = load_tci()                                    # 210 toric C.I. pairs

print(f"KS {len(ks)} | wp4 {len(wp)} | IPWS {len(toric)} | TCI {len(tci)} | "
      f"Davies {len(davies)}")

KS 30108 | wp4 2780 | IPWS 10237 | TCI 210 | Davies 30389


## 3. The Seven Totals — Reproducing the Research

Headline facts verified from the raw files (all reproduced below):

- **KS gap:** toric totals run 22…502; totals 2–21, 23, 24, 27 have no toric pair;
- **self-mirror diagonal:** 136 pairs with $h^{1,1}=h^{2,1}$ (χ = 0), contiguous
  $h=14\ldots 99$ below 100; **first pair (14,14) at total 28**;
- **per-target counts:** 17→3 known (0 toric), 28→17 (7), 29→8 (6), 66→63 (61),
  80→77 (75), 81→76 (76!), 92→89 (89!);
- **100%-toric totals below 85:** only {25, 75, 76, 81} — 81 is the largest;
- **the non-toric ladders** $h^{1,1}=1\ldots5$ die out: last hits (1,129)@130,
  (2,112)@114, (3,113)@116, (4,95)@99, (5,102)@107 — a total is 100% toric iff
  missed by all five at once.

In [3]:
# §3 — headline statistics of the total h11+h21 (grounded in the raw files)
def chi(h, g):
    return 2 * (h - g)

TARGETS = [17, 28, 29, 66, 80, 81, 92]

tot = collections.Counter(h + g for h, g in ks)
print("KS total h11+h21: min =", min(tot), " max =", max(tot))
print("KS gap (totals 2..40 with no toric pair):",
      [t for t in range(2, 41) if t not in tot])

sm = sorted(h for h, g in ks if h == g)
print(f"self-mirror pairs: {len(sm)}; first ({sm[0]},{sm[0]}) at total {2*sm[0]}; "
      f"contiguous h=14..99: {sm[:86] == list(range(14, 100))}")

print("\nper-target totals (known = Davies zoo, toric = alltoric.spec):")
for t in TARGETS:
    pairs = sorted(p for p in davies if sum(p) == t)
    nks = sum(1 for p in pairs if p in ks)
    smp = [p for p in pairs if p[0] == p[1]]
    print(f"  total {t:3d}: {len(pairs):3d} known | {nks:3d} toric | "
          f"self-mirror: {smp or '-'}")

full100 = [t for t in range(22, 131)
           if (ps := [p for p in davies if sum(p) == t]) and all(p in ks for p in ps)]
print("\n100%-toric totals 22..130:", full100)
print("100%-toric totals BELOW 85:", [r for r in full100 if r < 85])

print("\nlast non-toric pair per h11 ladder (1..5):")
for h in range(1, 6):
    nt = sorted((h, g) for (h0, g) in davies if h0 == h and (h, g) not in ks)
    if nt:
        print(f"  h11={h}: last non-toric {nt[-1]} at total {sum(nt[-1])}")

KS total h11+h21: min = 22  max = 502
KS gap (totals 2..40 with no toric pair): [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 24, 27]
self-mirror pairs: 136; first (14,14) at total 28; contiguous h=14..99: True

per-target totals (known = Davies zoo, toric = alltoric.spec):
  total  17:   3 known |   0 toric | self-mirror: -
  total  28:  17 known |   7 toric | self-mirror: [(14, 14)]
  total  29:   8 known |   6 toric | self-mirror: -
  total  66:  63 known |  61 toric | self-mirror: [(33, 33)]
  total  80:  77 known |  75 toric | self-mirror: [(40, 40)]
  total  81:  76 known |  76 toric | self-mirror: -
  total  92:  89 known |  89 toric | self-mirror: [(46, 46)]



100%-toric totals 22..130: [25, 75, 76, 81, 85, 86, 88, 89, 91, 92, 93, 94, 95, 96, 97, 100, 101, 103, 105, 106, 108, 109, 110, 111, 112, 113, 115, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129]
100%-toric totals BELOW 85: [25, 75, 76, 81]

last non-toric pair per h11 ladder (1..5):
  h11=1: last non-toric (1, 129) at total 130
  h11=2: last non-toric (2, 112) at total 114
  h11=3: last non-toric (3, 113) at total 116
  h11=4: last non-toric (4, 95) at total 99
  h11=5: last non-toric (5, 102) at total 107


## 4. Build the Calabi–Yau Knowledge Graph

`tools/build_cy3.py` seeds a typed, directed graph from the research folder:

- **paper nodes** — all 40 references (25 PDFs + cited classics + moment/SDP
  literature), each with `arxiv`/`pdf`/`txt` paths;
- **concept nodes** — constructions (CICY, TCI, conifold, quotients, Siegel,
  double octics, …), datasets, and mechanisms (KS gap, self-mirror diagonal,
  non-toric ladders, toric completeness, self-eigen presentations, Siegel echo, …);
- **total nodes** — the seven targets with their verified verdicts;
- **pair nodes** — the famous manifolds ((1,16) tri-cubic, (1,65) four-quadrics,
  (14,14), (28,0), (46,46), the quintic mirror, …);
- **~130 typed edges** — papers→concepts `describes/constructs`, totals→pairs
  `contains`, datasets→concepts `realizes/exhibits`, citations `cites`;
- **encoder layer** — the 20 extracted paper texts are chunked (1,200 chars,
  200 overlap), embedded and indexed (2,289 chunks) under their paper nodes.

The graph is persisted to **`graph_data/cy3/`** (separate from the default
GraphRAG graph) with §4.3a bidirectional consistency and is idempotent — safe
to re-run.

In [4]:
# §4 — seed the Calabi–Yau graph (idempotent; ~2,300 chunks)
t0 = time.time()
graph, encoder = build_cy3_graph()
print(graph.summary())
print(f"chunks indexed : {encoder.index.size()}")
print(f"consistency    : {len(graph.validate_consistency())} violations (§4.3a)")
print(f"components     : {len(graph.connected_components())}")
print(f"build time     : {time.time()-t0:.1f}s")
print(f"saved to       : {graph.path}")

KnowledgeGraph: 95 nodes, 144 edges
density=0.0323  components=1
  pagerank[Kreuzer–Skarke Dataset (alltoric.spec)]=0.0341
  pagerank[Machine Learning on the CY Landscape]=0.0322
  pagerank[Self-Mirror Diagonal (h11 = h21)]=0.0272
  pagerank[Moment Problem / SDP Toolbox]=0.0271
  pagerank[One-Parameter World (h11 = 1)]=0.0243
  pagerank[Total Hodge Number 28]=0.0235
chunks indexed : 2292
consistency    : 0 violations (§4.3a)
components     : 1
build time     : 1.8s
saved to       : D:\Deepin\Programming\20260720 GraphRAG\graph_data\cy3\knowledge_graph.json


## 5. Local Graphs — Depth-3 Working Memory

Every node materializes its **depth-3 ego network** — the bounded working
memory the agents operate on. Anchors below: a total, a concrete manifold, a
construction family and a dataset.

In [5]:
# §5 — local graphs L3(u) for four CY3 anchors
for anchor in ["total_66", "pair_1_65", "siegel_threefold", "ks_dataset"]:
    local = graph.materialize_local(anchor, depth=3)
    print(f"\n=== L3({graph.get_node(anchor).entryname}) ===")
    print("  stats:", local.stats)
    print("  nodes:", [n.entryname for n in list(local.nodes.values())[:12]])
    print("  sample path:", [graph.get_node(n).entryname
                             for n in (local.paths[0] if local.paths else [])])


=== L3(Total Hodge Number 66) ===
  stats: {'n': 63, 'm': 92, 'density': 0.047107014848950336, 'diameter': 3}
  nodes: ['The Kreuzer–Skarke Axiverse (Goodsell–Ringwald 2018)', 'The 24-Cell and Calabi–Yau Threefolds with Hodge Numbers (1,1) (Braun 2012)', 'Primitive contractions of Calabi–Yau threefolds II (Kapustka 2009)', 'Open Question: Are the multiplicities of (33,33), (40,40), (46,46) special?', 'Toric complete intersections and weighted projective space (Kreuzer–Riegler–Sahakyan 2003)', 'Planckian Distribution of the Total', 'Complete Intersection CYs (CICY)', '(28,0) — rigid Siegel quotient', 'Tables of Calabi–Yau equations (Almkvist–van Enckevort–van Straten–Zudilin 2005)', '(251,251) — topmost self-mirror', 'Topological string amplitudes, complete intersection Calabi–Yau spaces and threshold corrections (Klemm–Kreuzer–Riegler–Scheidegger 2005)', 'Total Hodge Number 66']
  sample path: ['Total Hodge Number 66', 'Self-Mirror Diagonal (h11 = h21)', 'Kreuzer–Skarke Dataset (allto

## 6. Encoder Layer — Vector RAG over the 20 Extracted Papers

Chunk → embed → index → hybrid search. Queries below hit the actual paper
texts (Freitag–Salvati Manni, Green–Hübsch–Lütken, He–Jejjala–Pontiggia, …).

In [6]:
# §6 — semantic search over the CY3 corpus
queries = [
    "complete intersection of four quadrics in P7 with Hodge numbers (1,65)",
    "Siegel modular threefold rigid quotient Euler number 92",
    "first self-mirror Hodge pair (14,14) with chi=0",
    "Planckian blackbody distribution of the total Hodge number",
]
for q in queries:
    print(f"\nQ: {q}")
    for chunk, sim in encoder.search(q, k=3):
        print(f"   {chunk.chunk_id} [{chunk.section[:48]}] "
              f"sim={sim:.3f}: {chunk.text[:76]}...")

print("\nhybrid node ranking for 'self-mirror diagonal' anchored at total_28:")
for nid, score in encoder.hybrid_search("self-mirror diagonal", graph,
                                        "total_28", k=6):
    print(f"   {graph.get_node(nid).entryname}  score={score:.3f}")


Q: complete intersection of four quadrics in P7 with Hodge numbers (1,65)
   vec://green_hubsch_lutken/meta [__meta__] sim=0.649: All Hodge Numbers of All Complete Intersection Calabi–Yau Manifolds (Green–H...
   vec://davies_zoo/c0096 [The Expanding Zoo of Calabi–Yau Threefolds (Davi] sim=0.492: (2010) 383–466, arXiv:0809.4681 [hep-th].
[3] P. Green and T. Hubsch, “Calab...
   vec://he_landscape/c1616 [The Calabi–Yau Landscape: from Geometry, to Phys] sim=0.472: generalization of
a hypersurface is a complete intersection, where the codim...

Q: Siegel modular threefold rigid quotient Euler number 92
   vec://pair_15_2/meta [__meta__] sim=0.560: (15,2) — Siegel modular threefold quotient X̂/Z3²; chi = +26; total 17. Frei...
   vec://siegel_threefold/meta [__meta__] sim=0.505: Siegel Modular Threefolds Freitag–Salvati Manni quotients: (15,2) at total 1...
   vec://freitag_salvati_siegel/meta [__meta__] sim=0.438: On Siegel threefolds with a projective Calabi–Yau model (Freitag–Salvati 

## 7. The DeepSeek Agent Operating on a Node (IPP loop)

`NodeAgent.operate` grounds a task in $\mathcal{L}_3(u)$ + encoder evidence and
runs the DeepSeek agent loop (function calling → four-phase tool pipeline →
answer). If the API is unreachable, a deterministic `MockProvider` keeps the
pipeline demonstrable.

In [7]:
# §7 — NodeAgent on total 66 (live DeepSeek, mock fallback)
try:
    llm = DeepSeekProvider(model=Config.get_model())
    llm.chat([{"role": "user", "content": "Reply: OK"}], max_tokens=8)
    provider_name = f"deepseek:{llm.model}"
except Exception as exc:
    llm = MockProvider()
    provider_name = f"mock (offline fallback): {exc}"
print("provider:", provider_name)

node_agent = NodeAgent(graph, encoder, llm=llm)
t0 = time.time()
result = node_agent.operate(
    "total_66",
    "Explain what makes total Hodge number 66 special: which concrete "
    "manifolds live on it, which constructions fill it, and how does it "
    "compare with totals 80, 81 and 92?",
)
print("\n=== ANSWER ===")
print(result["answer"][:1800])
print("\n=== WORKING MEMORY ===", result["local_graph"])
print("=== TOOLS CALLED ===", [e["tool"] for e in result["trace"] if e["tool"]])
print(f"=== run time: {time.time()-t0:.1f}s, tokens: {result['tokens']}")

provider: deepseek:deepseek-v4-flash



=== ANSWER ===
I now have a comprehensive understanding of the local graph. Let me compile my answer.

## What makes Total Hodge Number 66 special

**Total Hodge Number 66** is flagged **SPECIAL** in the graph, and the reason is a striking asymmetry: its *generic* self-mirror pair is unremarkable, but it hosts a genuinely exceptional one-parameter manifold.

### Concrete manifolds living on total 66

1. **(33,33) — the self-mirror weighted-P4 hypersurface** (χ = 0). This is the *generic* occupant — every even total 28–198 carries a self-mirror pair, so (33,33) is not itself special. It has **29 IPWS realizations**, including the self-eigen presentation **P(3,12,14,15,22)[66]** (degree = total) and a P:3 model with three K3 fibrations. Its polytope multiplicity is mid-range (max is (95,95) = 95).

2. **(1,65) — the four-quadrics CICY** [2 2 2 2] ⊂ P⁷ (χ = −128). This is the *exceptional* occupant and the real source of total 66's specialness. It is:
   - The **only plain-CICY one-param

## 8. Growth Agent — Recursive Self-Improvement (Layer 4)

`GrowthAgent.expand` probes gaps in $\mathcal{L}_3(u)$, asks DeepSeek for a
**structured JSON growth proposal** grounded in the local graph + encoder
evidence, then applies it through the graph tools with **dedup**, **per-run
limits** and §4.3a consistency, logging a version-control run entry.

The topic below is the research notebook's **open question**: are the polytope
multiplicities of the self-mirror pairs (33,33), (40,40), (46,46) among the
473,800,776 reflexive polytopes special? (Offline `MockProvider` yields an
empty proposal — run with a live DeepSeek key for real growth.)

In [8]:
# §8 — growth: propose + apply the open multiplicity question
growth = GrowthAgent(graph, encoder, llm=llm)
before = len(graph._nodes)
t0 = time.time()
growth_result = growth.expand(
    "ks_dataset",
    "Polytope multiplicities",
    "The open question of the research: are the polytope multiplicities of the "
    "self-mirror pairs (33,33), (40,40), (46,46) among the 473,800,776 "
    "reflexive polytopes special? Add a node capturing this open statistic.",
)
print("=== PROPOSAL ===")
print(json.dumps(growth_result["proposal"], ensure_ascii=False, indent=1)[:900])
print("\n=== APPLIED NODES ===")
for n in growth_result["applied_nodes"]:
    print("  +", n["node_id"], "-", n["entryname"])
print("=== APPLIED EDGES ===")
for e in growth_result["applied_edges"][:10]:
    print("  ", e["source"], "--[" + e["relation"] + "]-->", e["target"])
print("\nskipped  :", growth_result["skipped"])
print("errors   :", growth_result["errors"])
print("violations:", growth_result["consistency_violations"])
print(f"run time : {time.time()-t0:.1f}s, nodes {before} -> {len(graph._nodes)}")
if not growth_result["applied_nodes"]:
    print("\n(no nodes applied — offline MockProvider returns no JSON proposal; "
          "run with a live DeepSeek key for real growth proposals)")

relation 'includes' not in vocab; adding anyway


=== PROPOSAL ===
{
 "new_nodes": [
  {
   "node_id": "multiplicity_33_40_46",
   "entryname": "Multiplicity of self-mirror pairs (33,33), (40,40), (46,46)",
   "category": "concept",
   "description": "The specific polytope multiplicities of the self-mirror Hodge pairs (33,33), (40,40), (46,46) within the 473,800,776 reflexive 4-polytopes of the Kreuzer–Skarke dataset. The question of whether these multiplicities are special is open.",
   "links": [
    {
     "source": "multiplicity_33_40_46",
     "target": "Open Question: Are the multiplicities of (33,33), (40,40), (46,46) special?",
     "relation": "describes"
    },
    {
     "source": "multiplicity_33_40_46",
     "target": "Polytope Multiplicities",
     "relation": "part_of"
    },
    {
     "source": "multiplicity_33_40_46",
     "target": "(33,33) — self-mirror weighted-P4 hypersurface",
     "relation": "related_to"
    },
    {
     "sourc

=== APPLIED NODES ===
  + multiplicity_33_40_46 - Multiplicity of self-mirror pai

## 9. Export & the Web Control Center

- **Backward-compatible export** (ScientificInfrastructure `structurelist.json`
  + per-node `input.json`/`output.json`) into `graph_data/cy3/export/`;
- **Web UI** — serve the CY3 graph in the control center:

```bash
python ui/server.py --graph graph_data/cy3     # → http://127.0.0.3:8000
python ui/server.py                           # → default GraphRAG graph
```

Everything works on the CY3 graph: Graph/Search tabs, Agent tab
(`codex_RAG`/`codex_growth` bound to this graph), `/api/graph/local/<id>`,
`/api/search`, `/api/agent/node`, `/api/agent/grow`, and the VCL run log at
`/api/runs`.

In [9]:
# §9 — backward-compatible export + run log
out_root = graph.path.parent / "export"
export_backward_compatible(graph, out_root=out_root)
reg = json.loads((out_root / "structurelist.json").read_text(encoding="utf-8"))
print(f"export -> {out_root}  ({len(reg)} registry entries)")
print(json.dumps(reg[:4], ensure_ascii=False, indent=1))

runs = graph.runs()
print("\nlast run log (VCL):")
print(json.dumps(runs[-1], ensure_ascii=False, indent=1)[:700]
      if runs else "(no runs yet)")

export -> D:\Deepin\Programming\20260720 GraphRAG\graph_data\cy3\export  (98 registry entries)
[
 {
  "folderid": "almkvist_hypergeometric",
  "entryname": "Tables of Calabi–Yau equations (Almkvist–van Enckevort–van Straten–Zudilin 2005)"
 },
 {
  "folderid": "batyrev_kreuzer_conifold",
  "entryname": "Constructing new Calabi–Yau 3-folds and their mirrors via conifold transitions (Batyrev–Kreuzer 2010)"
 },
 {
  "folderid": "berglund_genetic",
  "entryname": "New Calabi–Yau Manifolds from Genetic Algorithms (Berglund–He–Heyes–… 2023)"
 },
 {
  "folderid": "bini_favale",
  "entryname": "Groups acting freely on Calabi–Yau threefolds embedded in a product of del Pezzo surfaces (Bini–Favale 2012)"
 }
]

last run log (VCL):
{
 "agent": "growth",
 "anchor": "ks_dataset",
 "topic": "Polytope multiplicities",
 "proposal": {
  "new_nodes": [
   {
    "node_id": "multiplicity_33_40_46",
    "entryname": "Multiplicity of self-mirror pairs (33,33), (40,40), (46,46)",
    "category": "concept",
   

## 10. Note Database — Living Markdown Notes per Node

Every graph node is also a **Markdown note** in the project store
(`database/calabiyau3fold/`): YAML front-matter, description, `[[wikilinks]]`
from the typed edges, paper/dataset metadata in `## Content`, and a
**Version Control Log** appended on every save (§4.4a). The notes are the
"growth store" — the knowledge accumulates as living documents, editable in
the Database tab of the control center.

**Assets:** every non-Markdown file the graph references (paper PDFs, pypdf
extractions, dataset files, the research notebook + HTML export) is copied
into the project's `assets/` folder, and the **node → file relationship** is
recorded in `assets/manifest.json` (role, path, size, sha256) plus a readable
`assets/README.md` table.

```text
database/calabiyau3fold/
  project.json              ← metadata + node/edge counts
  nodes/*.md                ← one note per graph node (95 notes)
  interactive.html          ← exported PyVis-style interactive graph
  assets/
    manifest.json           ← node_id → files (role · path · size · sha256)
    README.md               ← human-readable node ↔ file table
    papers/*.pdf            ← 21 paper PDFs
    extracted/*.txt         ← 21 pypdf extractions (encoder corpus)
    datasets/*              ← Kreuzer–Skarke / Davies / TCI data files
    research/*              ← the research notebook + HTML export
```


In [10]:
# §10 — sync the graph into the note database project (idempotent)
from database.notes import NoteStore
from ui.visuals import interactive_html
from tools.build_cy3 import sync_project_assets

store = NoteStore()
try:
    meta = store.create_project(
        "CalabiYau3fold",
        "Calabi–Yau threefold landscape: the seven totals, constructions, "
        "datasets and famous manifolds.")
    print("created project:", meta["slug"])
except ValueError:
    store.open_project("CalabiYau3fold")
    print("project already exists; opened:", store.current())

# 1) one .md note per graph node (YAML front-matter + [[wikilinks]] + VCL)
res = store.sync_from_graph(graph)
print("sync:", res)

# 2) copy every referenced file into the project's assets/ and write the
#    node → file relationship (manifest.json + README.md). Returns
#    {node_id: [{role, file (project-relative), size, sha256}]}.
manifest = sync_project_assets(
    graph, store.project_dir,
    research_files=[
        CY3_ROOT / "20260805 Calabi Yau Threefold.ipynb",
        CY3_ROOT / "20260805 Calabi Yau Threefold.html",
    ])

# 3) enrich paper/dataset notes with metadata; file paths are project-relative
enriched = 0
for nid, node in graph._nodes.items():
    c = node.content or {}
    lines = []
    if c.get("arxiv"):
        lines.append(f"- **arXiv:** {c['arxiv']}")
    for e in manifest.get(str(nid), []):
        if e["role"] == "paper":
            lines.append(f"- **Paper PDF:** `{e['file']}`")
        elif e["role"] == "extracted-text":
            lines.append(f"- **Extracted text (pypdf):** `{e['file']}`")
        elif e["role"] == "dataset":
            lines.append(f"- **Data file:** `{e['file']}`")
    if not lines:
        continue
    try:
        note = store.get_note(str(nid))
    except FileNotFoundError:
        continue
    note.content = "\n".join(lines)
    store.save_note(note, author="graph-build",
                    summary="Added source metadata (arXiv, asset file paths).")
    enriched += 1

# 4) PyVis-style interactive export into the project folder
(store.project_dir / "interactive.html").write_text(
    interactive_html(graph, title="Calabi–Yau Threefold Landscape"),
    encoding="utf-8")

store.open_project("CalabiYau3fold")
print("project  :", store.meta["name"], f"({store.meta['nodes']} notes, "
      f"{store.meta['edges']} links)")
print("notes    :", len(list(store.project_dir.glob('nodes/*.md'))))
print("enriched :", enriched, "notes with source metadata")
print("manifest :", len(manifest), "nodes mapped ->",
      sum(len(v) for v in manifest.values()), "files")
print("assets   :", len(list(store.project_dir.glob('assets/**/*'))),
      "entries under assets/")
print("interactive.html:", (store.project_dir / "interactive.html").stat().st_size, "bytes")

project already exists; opened: calabiyau3fold
sync: {'created': 3, 'updated': 95, 'skipped': 0}


project  : CalabiYau3fold (98 notes, 156 links)
notes    : 98
enriched : 45 notes with source metadata
manifest : 31 nodes mapped -> 55 files
assets   : 61 entries under assets/
interactive.html: 46266 bytes


---

## Version Control Log

Per ScientificInfrastructure §4.4a.

### v1.2 — 2026-08-07 (project assets + node ↔ file manifest)

- **New §10 asset pipeline** — every non-Markdown file the graph references
  is copied into `database/calabiyau3fold/assets/`:
  `papers/` (21 PDFs) · `extracted/` (21 pypdf texts) · `datasets/` (12 files)
  · `research/` (the research notebook + HTML export).
- **`assets/manifest.json`** records the **node → file relationship** for
  every node (role, project-relative path, size, sha256) and
  **`assets/README.md`** renders it as a readable table.
- Notes now reference **project-relative asset paths** (portable) instead of
  absolute source paths.

### v1.1 — 2026-08-07 (note database project)

- **New §10** — the graph is synced into the note database:
  `database/calabiyau3fold/` with **one `.md` note per node** (95 notes),
  YAML front-matter, `[[wikilinks]]` from the typed edges, paper/dataset
  metadata in `## Content` (arXiv / PDF / extracted text / data files),
  version control log per note, and an exported `interactive.html`.
- The Database tab of the control center opens this project by default.

### v1.0 — 2026-08-07 (CY3 graph conversion)

- **New `tools/build_cy3.py`** — seeds a 92-node / 133-edge knowledge graph
  from `assets/20260806 CalabiYau3fold/ResearchReferences/` (40 papers, 24
  concepts, 5 datasets, 7 target totals, 16 famous pairs; 2,289 encoder
  chunks from the 20 extracted texts). Persists to `graph_data/cy3/` — the
  default GraphRAG graph is untouched.
- **New notebook** — dataset loading with fixed paths (original `ROOT` was
  broken), headline statistics, graph build, local graphs, vector RAG,
  NodeAgent + GrowthAgent demos, backward-compatible export.
- **`ui/server.py --graph DIR`** — serve any custom graph folder
  (`knowledge_graph.json` + `vectors/index.json`); rebuild button respects it.
- Research verdicts (17/28/66/81 special, 29 mild, 80 generic, 92
  notable-but-general) encoded into the total-node descriptions.
- The `apply_*.py` fixer scripts in the source folder were not converted
  (one-off patch scripts against the research notebook's markdown).
